# QuantJourney SDK - Technical Analysis & Visualization

This notebook demonstrates quantitative technical analysis:
- Moving averages (SMA, EMA)
- Bollinger Bands
- RSI (Relative Strength Index)
- MACD
- Volatility analysis
- Risk metrics (Sharpe, Max Drawdown, VaR)

**API:** https://api.quantjourney.cloud

## Run Output

![05_technical_analysis](../plots/05_technical_analysis_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# API Key authentication (recommended)
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


In [ ]:
# Helper functions
def calculate_sma(prices, window):
    return prices.rolling(window=window).mean()

def calculate_ema(prices, window):
    return prices.ewm(span=window, adjust=False).mean()

def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_bollinger_bands(prices, window=20, num_std=2):
    sma = prices.rolling(window=window).mean()
    std = prices.rolling(window=window).std()
    upper = sma + (std * num_std)
    lower = sma - (std * num_std)
    return sma, upper, lower


## 1. Fetch Historical Data

In [ ]:
# Fetch AAPL data (2 years for good analysis)
symbol = "AAPL"

response = qj.eod.get_historical_prices(
    symbol=symbol,
    start_date="2023-01-01",
    end_date="2024-12-31",
    frequency="1d"
)
prices = response.get('value', response) if isinstance(response, dict) else response

df = pd.DataFrame(prices)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"Symbol: {symbol}")
print(f"Records: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.tail()


## 2. Moving Averages Analysis

In [ ]:
# Calculate moving averages
df['SMA_20'] = calculate_sma(df['close'], 20)
df['SMA_50'] = calculate_sma(df['close'], 50)
df['SMA_200'] = calculate_sma(df['close'], 200)
df['EMA_12'] = calculate_ema(df['close'], 12)
df['EMA_26'] = calculate_ema(df['close'], 26)

# Plot
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=df['date'],
    open=df['open'], high=df['high'],
    low=df['low'], close=df['close'],
    name='Price'
))

fig.add_trace(go.Scatter(x=df['date'], y=df['SMA_20'], name='SMA 20', line=dict(width=1)))
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA_50'], name='SMA 50', line=dict(width=1.5)))
fig.add_trace(go.Scatter(x=df['date'], y=df['SMA_200'], name='SMA 200', line=dict(width=2)))

fig.update_layout(
    title=f'{symbol} with Moving Averages',
    yaxis_title='Price ($)',
    xaxis_rangeslider_visible=False,
    template='plotly_dark',
    height=600
)
fig.show()

# Trend analysis
latest = df.iloc[-1]
print(f"\nTrend Analysis:")
print(f"  Price: ${latest['close']:.2f}")
print(f"  vs SMA 20: {'Above ↑' if latest['close'] > latest['SMA_20'] else 'Below ↓'}")
print(f"  vs SMA 50: {'Above ↑' if latest['close'] > latest['SMA_50'] else 'Below ↓'}")
print(f"  vs SMA 200: {'Above ↑' if latest['close'] > latest['SMA_200'] else 'Below ↓'}")


## 3. Bollinger Bands

In [ ]:
# Calculate Bollinger Bands
df['BB_middle'], df['BB_upper'], df['BB_lower'] = calculate_bollinger_bands(df['close'])

# Plot
fig = go.Figure()

# Bollinger Bands fill
fig.add_trace(go.Scatter(
    x=pd.concat([df['date'], df['date'][::-1]]),
    y=pd.concat([df['BB_upper'], df['BB_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(100,100,100,0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='BB Range'
))

fig.add_trace(go.Scatter(x=df['date'], y=df['BB_upper'], name='Upper Band', line=dict(dash='dash', width=1)))
fig.add_trace(go.Scatter(x=df['date'], y=df['BB_lower'], name='Lower Band', line=dict(dash='dash', width=1)))
fig.add_trace(go.Scatter(x=df['date'], y=df['BB_middle'], name='Middle Band', line=dict(width=1)))
fig.add_trace(go.Scatter(x=df['date'], y=df['close'], name='Price', line=dict(width=2, color='cyan')))

fig.update_layout(
    title=f'{symbol} Bollinger Bands',
    yaxis_title='Price ($)',
    template='plotly_dark',
    height=500
)
fig.show()

# Position analysis
latest = df.iloc[-1]
bb_position = (latest['close'] - latest['BB_lower']) / (latest['BB_upper'] - latest['BB_lower']) * 100
print(f"\nBollinger Bands Position: {bb_position:.0f}% (0=oversold, 100=overbought)")


## 4. RSI (Relative Strength Index)

In [ ]:
# Calculate RSI
df['RSI'] = calculate_rsi(df['close'], 14)

# Plot with price
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.05
)

# Price
fig.add_trace(
    go.Scatter(x=df['date'], y=df['close'], name='Price'),
    row=1, col=1
)

# RSI
fig.add_trace(
    go.Scatter(x=df['date'], y=df['RSI'], name='RSI', line=dict(color='purple')),
    row=2, col=1
)

# Overbought/Oversold zones
fig.add_hline(y=70, line_dash='dash', line_color='red', row=2, col=1)
fig.add_hline(y=30, line_dash='dash', line_color='green', row=2, col=1)
fig.add_hline(y=50, line_dash='dot', line_color='gray', row=2, col=1)

# Add shaded regions
fig.add_hrect(y0=70, y1=100, fillcolor='red', opacity=0.1, row=2, col=1)
fig.add_hrect(y0=0, y1=30, fillcolor='green', opacity=0.1, row=2, col=1)

fig.update_layout(
    title=f'{symbol} RSI Analysis',
    template='plotly_dark',
    height=600
)
fig.update_yaxes(title_text='Price ($)', row=1, col=1)
fig.update_yaxes(title_text='RSI', row=2, col=1, range=[0, 100])
fig.show()

latest_rsi = df['RSI'].iloc[-1]
if latest_rsi > 70:
    signal = "OVERBOUGHT"
elif latest_rsi < 30:
    signal = "OVERSOLD"
else:
    signal = "Neutral"
print(f"\nCurrent RSI: {latest_rsi:.1f} - {signal}")


## 5. MACD

In [ ]:
# Calculate MACD (need to add EMA_12 and EMA_26 first)
df['EMA_12'] = calculate_ema(df['close'], 12)
df['EMA_26'] = calculate_ema(df['close'], 26)
df['MACD'] = df['EMA_12'] - df['EMA_26']
df['MACD_signal'] = calculate_ema(df['MACD'], 9)
df['MACD_histogram'] = df['MACD'] - df['MACD_signal']

# Plot
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.05
)

# Price
fig.add_trace(
    go.Scatter(x=df['date'], y=df['close'], name='Price'),
    row=1, col=1
)

# MACD
colors = ['green' if h >= 0 else 'red' for h in df['MACD_histogram']]
fig.add_trace(
    go.Bar(x=df['date'], y=df['MACD_histogram'], name='Histogram', marker_color=colors),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df['date'], y=df['MACD'], name='MACD', line=dict(color='blue')),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df['date'], y=df['MACD_signal'], name='Signal', line=dict(color='orange')),
    row=2, col=1
)

fig.update_layout(
    title=f'{symbol} MACD Analysis',
    template='plotly_dark',
    height=600
)
fig.update_yaxes(title_text='Price ($)', row=1, col=1)
fig.update_yaxes(title_text='MACD', row=2, col=1)
fig.show()

latest = df.iloc[-1]
macd_signal = "BULLISH" if latest['MACD'] > latest['MACD_signal'] else "BEARISH"
print(f"\nMACD Signal: {macd_signal}")


## 6. Volatility Analysis

In [ ]:
# Calculate returns and volatility
df['return'] = df['close'].pct_change()
df['volatility_20d'] = df['return'].rolling(20).std() * np.sqrt(252) * 100
df['volatility_60d'] = df['return'].rolling(60).std() * np.sqrt(252) * 100

# Plot
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('Price', 'Annualized Volatility'),
    vertical_spacing=0.1
)

fig.add_trace(
    go.Scatter(x=df['date'], y=df['close'], name='Price'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df['date'], y=df['volatility_20d'], name='20-day Vol', fill='tozeroy'),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=df['date'], y=df['volatility_60d'], name='60-day Vol'),
    row=2, col=1
)

fig.update_layout(
    title=f'{symbol} Volatility Analysis',
    template='plotly_dark',
    height=600
)
fig.update_yaxes(title_text='Price ($)', row=1, col=1)
fig.update_yaxes(title_text='Volatility (%)', row=2, col=1)
fig.show()

print(f"\nVolatility Stats:")
print(f"  Current (20d): {df['volatility_20d'].iloc[-1]:.1f}%")
print(f"  Current (60d): {df['volatility_60d'].iloc[-1]:.1f}%")
print(f"  Average: {df['volatility_20d'].mean():.1f}%")
print(f"  Max: {df['volatility_20d'].max():.1f}%")


## 7. Risk Metrics

In [ ]:
# Calculate risk metrics
returns = df['return'].dropna()

# Annualized return
total_return = (df['close'].iloc[-1] / df['close'].iloc[0]) - 1
annual_return = (1 + total_return) ** (252 / len(df)) - 1

# Annualized volatility
annual_vol = returns.std() * np.sqrt(252)

# Sharpe Ratio (5% risk-free)
risk_free = 0.05
sharpe = (annual_return - risk_free) / annual_vol

# Maximum Drawdown
df['cummax'] = df['close'].cummax()
df['drawdown'] = (df['close'] - df['cummax']) / df['cummax'] * 100
max_drawdown = df['drawdown'].min()

# VaR (95%)
var_95 = np.percentile(returns, 5) * 100

# Plot drawdown
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('Price', 'Drawdown'),
    vertical_spacing=0.1
)

fig.add_trace(
    go.Scatter(x=df['date'], y=df['close'], name='Price'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df['date'], y=df['drawdown'], fill='tozeroy', 
               fillcolor='rgba(255,0,0,0.3)', name='Drawdown'),
    row=2, col=1
)

fig.update_layout(
    title=f'{symbol} Drawdown Analysis',
    template='plotly_dark',
    height=550
)
fig.update_yaxes(title_text='Price ($)', row=1, col=1)
fig.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig.show()

print(f"\nRisk Metrics Summary:")
print(f"  Annual Return:    {annual_return*100:+.1f}%")
print(f"  Annual Volatility: {annual_vol*100:.1f}%")
print(f"  Sharpe Ratio:     {sharpe:.2f}")
print(f"  Max Drawdown:     {max_drawdown:.1f}%")
print(f"  VaR (95% daily):  {var_95:.2f}%")


## 8. Technical Summary Dashboard

In [ ]:
# Summary dashboard
# Calculate SMA_200 if not exists
if 'SMA_200' not in df.columns:
    df['SMA_200'] = calculate_sma(df['close'], 200)

latest = df.iloc[-1]

print("="*60)
print(f"TECHNICAL ANALYSIS SUMMARY - {symbol}")
print("="*60)

print(f"\nPRICE")
print(f"   Current: ${latest['close']:.2f}")
sma_200 = latest.get('SMA_200', latest['close'])
print(f"   vs SMA 200: {'↑ Bullish' if latest['close'] > sma_200 else '↓ Bearish'}")

print(f"\nMOMENTUM")
print(f"   RSI (14): {latest['RSI']:.1f} {'Overbought' if latest['RSI'] > 70 else 'Oversold' if latest['RSI'] < 30 else 'Neutral'}")
print(f"   MACD: {'Bullish' if latest['MACD'] > latest['MACD_signal'] else 'Bearish'}")

print(f"\nVOLATILITY")
print(f"   Current (20d): {latest['volatility_20d']:.1f}%")
print(f"   Bollinger %B: {((latest['close'] - latest['BB_lower']) / (latest['BB_upper'] - latest['BB_lower']) * 100):.0f}%")

print(f"\nRISK")
print(f"   Sharpe Ratio: {sharpe:.2f}")
print(f"   Max Drawdown: {max_drawdown:.1f}%")

print("\n" + "="*60)


## Summary

Technical indicators implemented:
- **Moving Averages**: SMA (20, 50, 200), EMA (12, 26)
- **Bollinger Bands**: 20-day, 2 std dev
- **RSI**: 14-period
- **MACD**: 12, 26, 9 periods
- **Risk Metrics**: Sharpe, Max Drawdown, VaR